# NSys Profile Analysis

Load Nsight Systems `.sqlite` profiles and compute steady-state kernel timing metrics for each (GPU, kernel, seq_len, head_dim) configuration.

In [ ]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
PROFILES_DIR = Path("profiles")
NUM_ITERATIONS = 20  # --iterations 20 used in GPU_Metrics.ipynb nsys profiling commands

# CUPTI copyKind enum values
COPY_KIND_H2D = 1
COPY_KIND_D2H = 2

In [ ]:
def get_nvtx_timed_region(conn: sqlite3.Connection) -> tuple[int, int]:
    """Return (start_ns, end_ns) for the 'timed_region' NVTX annotation."""
    row = conn.execute("""
                       SELECT n.start, n.end
                       FROM NVTX_EVENTS n
                                LEFT JOIN StringIds s ON n.textId = s.id
                       WHERE COALESCE(n.text, s.value) = 'timed_region' LIMIT 1
                       """).fetchone()
    if row is None:
        raise ValueError("No 'timed_region' NVTX marker found")
    return row[0], row[1]


def has_kernel_table(conn: sqlite3.Connection) -> bool:
    row = conn.execute("""
                       SELECT name
                       FROM sqlite_master
                       WHERE type = 'table'
                         AND name = 'CUPTI_ACTIVITY_KIND_KERNEL'
                       """).fetchone()
    return row is not None


def load_kernel_rows(conn: sqlite3.Connection, timed_start: int, timed_end: int) -> pd.DataFrame:
    """Load all CUDA kernel launches within the timed_region window."""
    return pd.read_sql_query("""
                             SELECT k.start,
                                    k.end,
                                    (k.end - k.start) AS duration_ns,
                                    s.value           AS kernel_name,
                                    k.gridX,
                                    k.gridY,
                                    k.gridZ,
                                    k.blockX,
                                    k.blockY,
                                    k.blockZ,
                                    k.registersPerThread,
                                    k.staticSharedMemory,
                                    k.dynamicSharedMemory
                             FROM CUPTI_ACTIVITY_KIND_KERNEL k
                                      LEFT JOIN StringIds s ON k.demangledName = s.id
                             WHERE k.start >= ?
                               AND k.end <= ?
                             ORDER BY k.start
                             """, conn, params=(timed_start, timed_end))


def group_kernels_into_iterations(df: pd.DataFrame, num_iterations: int) -> tuple[pd.DataFrame, int]:
    """
    Assign each kernel launch to an iteration index and sum durations per iteration.

    Handles multi-kernel backends (e.g. math launches ~15 sub-kernels per attention call)
    by dividing the sorted kernel sequence into equal-sized chunks.
    Returns (per_iteration_durations_df, kernels_per_iteration).
    """
    total = len(df)
    kernels_per_iter = total // num_iterations
    if kernels_per_iter == 0:
        raise ValueError(f"Too few kernels ({total}) for {num_iterations} iterations")
    # Trim any extra kernels beyond the last complete iteration
    df = df.iloc[:kernels_per_iter * num_iterations].copy()
    df["iter_idx"] = np.arange(len(df)) // kernels_per_iter
    per_iter = df.groupby("iter_idx")["duration_ns"].sum().reset_index()
    return per_iter, kernels_per_iter


def load_memcpy_rows(conn: sqlite3.Connection, timed_start: int, timed_end: int) -> pd.DataFrame:
    """Load memory copy events within the timed_region window. Returns empty DataFrame if table absent."""
    tables = {r[0] for r in conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table'"
    ).fetchall()}
    if "CUPTI_ACTIVITY_KIND_MEMCPY" not in tables:
        return pd.DataFrame()
    return pd.read_sql_query("""
                             SELECT (end - start) AS duration_ns,
                                    bytes,
                                    copyKind
                             FROM CUPTI_ACTIVITY_KIND_MEMCPY
                             WHERE start >= ? AND end <= ?
                             """, conn, params=(timed_start, timed_end))


def get_gpu_device_info(conn: sqlite3.Connection) -> dict:
    row = conn.execute("""
                       SELECT name, clockRate, smCount, totalMemory
                       FROM TARGET_INFO_GPU LIMIT 1
                       """).fetchone()
    if row is None:
        return {}
    return {
        "gpu_full_name": row[0],
        "clock_rate_hz": row[1],
        "sm_count": row[2],
        "total_memory_bytes": row[3],
    }

In [ ]:
rows = []

for gpu_dir in sorted(PROFILES_DIR.iterdir()):
    if not gpu_dir.is_dir():
        continue
    gpu = gpu_dir.name

    for size_dir in sorted(gpu_dir.iterdir()):
        if not size_dir.is_dir():
            continue
        try:
            seq_str, dim_str = size_dir.name.split("x")
            seq_len, head_dim = int(seq_str), int(dim_str)
        except ValueError:
            continue

        for sqlite_path in sorted(size_dir.glob("*.sqlite")):
            kernel = sqlite_path.stem
            conn = sqlite3.connect(sqlite_path)
            try:
                timed_start, timed_end = get_nvtx_timed_region(conn)
                gpu_info = get_gpu_device_info(conn)

                row: dict = {
                    "gpu": gpu,
                    "gpu_full_name": gpu_info.get("gpu_full_name"),
                    "kernel": kernel,
                    "seq_len": seq_len,
                    "head_dim": head_dim,
                }

                if has_kernel_table(conn):
                    df_k = load_kernel_rows(conn, timed_start, timed_end)
                    per_iter, kpi = group_kernels_into_iterations(df_k, NUM_ITERATIONS)
                    durations = per_iter["duration_ns"]

                    row.update({
                        "duration_mean_ns": durations.mean(),
                        "duration_std_ns": durations.std(),
                        "duration_median_ns": durations.median(),
                        "duration_min_ns": durations.min(),
                        "duration_max_ns": durations.max(),
                        "kernels_per_iter": kpi,
                        "data_source": "kernel_table",
                    })

                    first = df_k.iloc[0]
                    row.update({
                        "grid_x": first["gridX"],
                        "grid_y": first["gridY"],
                        "grid_z": first["gridZ"],
                        "block_x": first["blockX"],
                        "block_y": first["blockY"],
                        "block_z": first["blockZ"],
                        "registers_per_thread": first["registersPerThread"],
                        "static_shared_mem_bytes": first["staticSharedMemory"],
                        "dynamic_shared_mem_bytes": first["dynamicSharedMemory"],
                    })

                    df_m = load_memcpy_rows(conn, timed_start, timed_end)
                    if not df_m.empty:
                        h2d = df_m[df_m["copyKind"] == COPY_KIND_H2D]
                        d2h = df_m[df_m["copyKind"] == COPY_KIND_D2H]
                        row.update({
                            "memcpy_h2d_bytes": h2d["bytes"].sum() if not h2d.empty else np.nan,
                            "memcpy_h2d_duration_ns": h2d["duration_ns"].sum() if not h2d.empty else np.nan,
                            "memcpy_d2h_bytes": d2h["bytes"].sum() if not d2h.empty else np.nan,
                            "memcpy_d2h_duration_ns": d2h["duration_ns"].sum() if not d2h.empty else np.nan,
                        })
                    else:
                        row.update({
                            "memcpy_h2d_bytes": np.nan,
                            "memcpy_h2d_duration_ns": np.nan,
                            "memcpy_d2h_bytes": np.nan,
                            "memcpy_d2h_duration_ns": np.nan,
                        })

                else:
                    # No per-kernel timing available; use NVTX total duration / iterations
                    avg_ns = (timed_end - timed_start) / NUM_ITERATIONS
                    row.update({
                        "duration_mean_ns": avg_ns,
                        "duration_std_ns": np.nan,
                        "duration_median_ns": np.nan,
                        "duration_min_ns": np.nan,
                        "duration_max_ns": np.nan,
                        "kernels_per_iter": np.nan,
                        "data_source": "nvtx_total",
                        "grid_x": np.nan, "grid_y": np.nan, "grid_z": np.nan,
                        "block_x": np.nan, "block_y": np.nan, "block_z": np.nan,
                        "registers_per_thread": np.nan,
                        "static_shared_mem_bytes": np.nan,
                        "dynamic_shared_mem_bytes": np.nan,
                        "memcpy_h2d_bytes": np.nan,
                        "memcpy_h2d_duration_ns": np.nan,
                        "memcpy_d2h_bytes": np.nan,
                        "memcpy_d2h_duration_ns": np.nan,
                    })

                rows.append(row)

            except Exception as e:
                print(f"ERROR {sqlite_path}: {e}")
            finally:
                conn.close()

df_nsys = pd.DataFrame(rows)
print(f"Loaded {len(df_nsys)} rows from {PROFILES_DIR}")
df_nsys.head(10)

In [ ]:
print("Rows by GPU and kernel:")
print(df_nsys.groupby(["gpu", "kernel"]).size().to_string())
print()
print("Data source breakdown:")
print(df_nsys["data_source"].value_counts().to_string())
print()
print("Columns:", list(df_nsys.columns))

In [ ]:
out_path = Path("metrics/all_metrics_nsys.csv")
df_nsys.to_csv(out_path, index=False)
print(f"Saved {len(df_nsys)} rows to {out_path}")

## Plot durations

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.facecolor":  "white",
    "axes.facecolor":    "white",
    "savefig.facecolor": "white",
    "text.color":        "black",
    "axes.labelcolor":   "black",
    "axes.edgecolor":    "black",
    "xtick.color":       "black",
    "ytick.color":       "black",
    "font.size":         12,
    "axes.titlesize":    16,
    "axes.labelsize":    14,
    "xtick.labelsize":   12,
    "ytick.labelsize":   12,
})

# convert kernel names to same labels as other plots
kernel_names = {
    "math": "Math",
    "flash": "FA2",
    "mea": "MEA",
    "cudnn": "cuDNN",
    "flash3": "FA3"
}

def plot_execution_times(
        gpu: str,
        seq_len: int,
        head_dim: int,
        df: pd.DataFrame = None,
        show: bool = False,
        save: bool = True,
) -> None:
    if df is None:
        df = df_nsys
    subset = df[(df["gpu"] == gpu) & (df["seq_len"] == seq_len) & (df["head_dim"] == head_dim)]
    if subset.empty:
        return

    means = (subset["duration_mean_ns"] / 1_000).tolist()  # ns → µs
    stds = (subset["duration_std_ns"] / 1_000).tolist()
    labels = subset["kernel"].map(lambda k: kernel_names[k]).tolist()

    math_rows = subset[subset["kernel"] == "math"]
    math_mean = (math_rows["duration_mean_ns"].iloc[0] / 1_000) if not math_rows.empty else None

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.bar(range(len(labels)), means, yerr=stds, capsize=4)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=30, ha="right")
    ax.set_ylabel("Duration (µs)")
    ax.set_title(f"{gpu.upper()} — SeqLen={seq_len}, HeadDim={head_dim}")

    label_offset = max(means) * 0.02
    for i, (mean, std) in enumerate(zip(means, stds)):
        pct_str = f"({mean / math_mean * 100:.1f}%)" if math_mean else ""
        std_str = f" ± {std:.1f}" if not np.isnan(std) else ""
        bar_label = f"{mean:.1f}{std_str}\n{pct_str}"
        top = mean + (std if not np.isnan(std) else 0) + label_offset
        ax.text(i, top, bar_label, ha="center", va="bottom", fontsize=10)

    ax.set_ylim(0, max(m + (s if not np.isnan(s) else 0) for m, s in zip(means, stds)) * 1.25)
    fig.tight_layout()

    if save:
        out = Path("plots/execution_times") / f"{gpu}_{seq_len}x{head_dim}.png"
        out.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(out)
    if show:
        plt.show()
    plt.close()

In [ ]:
for gpu in df_nsys["gpu"].unique():
    for seq_len in sorted(df_nsys["seq_len"].unique()):
        for head_dim in sorted(df_nsys["head_dim"].unique()):
            plot_execution_times(gpu, seq_len, head_dim)

In [ ]:
plot_execution_times("a100", 2048, 128, show=True, save=False)